#### Notes for later

In [1]:
import polars as pl
import plotly.express as px
import pandas as pd  
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter, date2num
import time
import csv

#### Load last month's data

In [2]:
last_month = time.localtime().tm_mon - 1

current_yr = time.localtime().tm_year

# handling if last month was January
if last_month == 0:
    last_month = 12
    current_yr = current_yr - 1


df = pl.read_csv(r'C:\Users\Denne\Documents\VSCodeProjects\FinancialStreamline2\BankStreamline\Data\DebitHistory.csv')

# Date work
df = df.with_columns(
                pl.col("Post Date").str.strptime(pl.Date, "%m/%d/%Y").alias("date")
                ).with_columns([
                pl.col("date").dt.month().alias("month"),
                pl.col("date").dt.day().alias("day"),
                pl.col("date").dt.year().alias("year")
                ]) \
        .filter((pl.col('month') == last_month) & 
                (pl.col('year') == current_yr)) \
        .with_columns(date = pl.concat_str([pl.col('year').cast(pl.Utf8), pl.lit('-'), pl.col('month').cast(pl.Utf8), pl.lit('-'), pl.col('day').cast(pl.Utf8)], separator="")) \
        .with_columns(monthName = pl.col("date").str.strptime(pl.Date, "%Y-%m-%d").dt.strftime("%b")) \
        .with_columns(weekDay = pl.col("date").str.strptime(pl.Date, "%Y-%m-%d").dt.strftime("%a")) \
        .with_columns(quarter = pl.col("date").str.strptime(pl.Date, "%Y-%m-%d").dt.quarter())
                        
# # cost column
df = df.with_columns(pl.col('Debit').fill_null(value=0)) \
        .with_columns(pl.col('Credit').fill_null(value=0)) \
        .with_columns(cost =pl.col('Debit') + pl.col('Credit')) \
        
# # the gate/clean up
df = df.filter(~pl.col('Description').str.contains('ELECTRONIC DEPOSIT MURPHY OIL USA PPAYROLL|Funds Transfer via Mobile')) \
        .filter(~pl.col('Description').str.contains('DEPOSIT|deposit|Deposit|WITHDRAWAL-CASH')) \
        .filter(~pl.col('Description').str.contains('Withdrawal LOAN PAYMENT TRANSFER')) \
        .rename({'Description': 'description'}) \
        .with_columns(cardType = pl.lit('Debit'))

print(len(df))
df.limit(3)

18


Account Number,Post Date,Check,description,Debit,Credit,Status,date,month,day,year,monthName,weekDay,quarter,cost,cardType
str,str,str,str,f64,f64,str,str,i8,i8,i32,str,str,i8,f64,str
"""***0530002""","""10/30/2026""",null,"""ACH Withdrawal SUMMIT SUA BILL…",101.0,0.0,"""Posted""","""2026-10-30""",10,30,2026,"""Oct""","""Fri""",4,101.0,"""Debit"""
"""***0530002""","""10/30/2026""",null,"""ACH Withdrawal El Dorado Water…",167.19,0.0,"""Posted""","""2026-10-30""",10,30,2026,"""Oct""","""Fri""",4,167.19,"""Debit"""
"""***0530002""","""10/26/2026""",null,"""ACH Withdrawal FID BKG SVC LLC…",723.15,0.0,"""Posted""","""2026-10-26""",10,26,2026,"""Oct""","""Mon""",4,723.15,"""Debit"""


### Categorize data

In [3]:
# pulls from the categorize py file

from Categorize import make_categories
import Categorize
import importlib
importlib.reload(Categorize)


df = make_categories(df)

df.limit(3)


date,category,description,cost,monthName,month,day,year,weekDay,cardType,quarter
str,str,str,f64,str,i8,i8,i32,str,str,i8
"""2026-10-30""","""natural gas""","""ACH Withdrawal SUMMIT SUA BILL…",101.0,"""Oct""",10,30,2026,"""Fri""","""Debit""",4
"""2026-10-30""","""water""","""ACH Withdrawal El Dorado Water…",167.19,"""Oct""",10,30,2026,"""Fri""","""Debit""",4
"""2026-10-26""","""misc""","""ACH Withdrawal FID BKG SVC LLC…",723.15,"""Oct""",10,26,2026,"""Mon""","""Debit""",4


### Look at last month's misc

In [4]:
list(df.filter(pl.col('category') == 'misc')['description'].unique())

['ACH Withdrawal NATL FIN SVC LLC ACCTVERIFY BLAKE DANIEL DENNETT',
 'ACH Withdrawal FID BKG SVC LLC MONEYLINE BLAKE DANIEL DENNETT',
 'ACH Withdrawal VENMO PAYMENT ADANYA AND BLAKE DENNE']

#### Make sure correct month/year is being loaded

In [5]:
print(list(df['month'].unique()))
print(list(df['monthName'].unique()))
print(list(df['weekDay'].unique()))
print(list(df['year'].unique()))
print(list(df['day'].unique()))
# print(df.select(pl.col("date").min()))
# print(df.select(pl.col("date").max())) # needs fixed, looking at str, not date

[10]
['Oct']
['Wed', 'Tue', 'Fri', 'Sun', 'Thu', 'Mon']
[2026]
[1, 2, 5, 7, 11, 12, 13, 14, 16, 21, 22, 23, 26, 30]


### Check for double water, power, or natural gas

In [6]:
df.filter((pl.col('category') == 'water') | (pl.col('category') == 'power') | (pl.col('category') == 'natural gas')).sort('category')

date,category,description,cost,monthName,month,day,year,weekDay,cardType,quarter
str,str,str,f64,str,i8,i8,i32,str,str,i8
"""2026-10-30""","""natural gas""","""ACH Withdrawal SUMMIT SUA BILL…",101.0,"""Oct""",10,30,2026,"""Fri""","""Debit""",4
"""2026-10-26""","""power""","""ACH Withdrawal ENTERGY ARKANSA…",100.49,"""Oct""",10,26,2026,"""Mon""","""Debit""",4
"""2026-10-30""","""water""","""ACH Withdrawal El Dorado Water…",167.19,"""Oct""",10,30,2026,"""Fri""","""Debit""",4
"""2026-10-1""","""water""","""ACH Withdrawal El Dorado Water…",76.52,"""Oct""",10,1,2026,"""Thu""","""Debit""",4


### Check number of rows (40ish per month)
Apr: 38
<br>May: 53
<br>June: 23 - Vegas trip
<br>May: 65
<br>Aug: 48
<br>Sep: 75
<br>Oct: 46
<br>Nov: 66
<br>Dec: 51
<br> 2025 - raising limit on credit card
<br>Jan: 34
<br>Feb: 34
<br>Mar: 15
<br>Apr: 36
<br>May: 18
<br>June: 27
<br>July: 16
<br>Aug: 32
<br>Sep: 17
<br>Oct: 18


In [8]:
len(df)

18

### Load the data (handle w/ caution)

In [ ]:
# path = r'C:\Users\Denne\Documents\VSCodeProjects\FinancialStreamline2\BankStreamline\Data\BankDataProd.csv'


# with open(path, mode="a", encoding="utf-8") as f:
#    df.write_csv(f, include_header=False)